In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [4]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5), (0.5))
    ])

transform = transforms.ToTensor()

mnist_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

data_loader = torch.utils.data.DataLoader(dataset=mnist_data,
                                          batch_size=64,
                                          shuffle=True)

In [5]:
dataiter = iter(data_loader)
images, labels = next(dataiter)
print(torch.min(images), torch.max(images))

tensor(0.) tensor(1.)


In [80]:
# repeatedly reduce the size
class Autoencoder_FullyConnected(nn.Module):
    def __init__(self):
        super().__init__()        
        self.encoder = nn.Sequential(
            nn.Linear(28 * 28, 128), # (N, 784) -> (N, 128)
            nn.SiLU(),
            nn.Linear(128, 32)
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(32, 128),
            nn.SiLU(),
            nn.Linear(128, 28 * 28),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

In [88]:
model = Autoencoder_FullyConnected()

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
#optimizer = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)

In [89]:
# Point to training loop with SilU (sigmoid Linear Unit) activation
num_epochs = 30
outputs = []
for epoch in range(num_epochs):
    for (img, _) in data_loader:
        img = img.reshape(-1, 28*28) # -> use for Autoencoder_FullyConnected for flattening [64,1,28,28] -> [64,784] 
        recon = model(img)
        loss = criterion(recon, img)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch:{epoch+1}, Loss:{loss.item():.4f}')
    outputs.append((epoch, img, recon))

Epoch:1, Loss:0.0206
Epoch:2, Loss:0.0183
Epoch:3, Loss:0.0149
Epoch:4, Loss:0.0109
Epoch:5, Loss:0.0135
Epoch:6, Loss:0.0122
Epoch:7, Loss:0.0101
Epoch:8, Loss:0.0088
Epoch:9, Loss:0.0098
Epoch:10, Loss:0.0103
Epoch:11, Loss:0.0094
Epoch:12, Loss:0.0085
Epoch:13, Loss:0.0082
Epoch:14, Loss:0.0105
Epoch:15, Loss:0.0092
Epoch:16, Loss:0.0077
Epoch:17, Loss:0.0091
Epoch:18, Loss:0.0088
Epoch:19, Loss:0.0102
Epoch:20, Loss:0.0089
Epoch:21, Loss:0.0075
Epoch:22, Loss:0.0068
Epoch:23, Loss:0.0105
Epoch:24, Loss:0.0086
Epoch:25, Loss:0.0085
Epoch:26, Loss:0.0091
Epoch:27, Loss:0.0088
Epoch:28, Loss:0.0077
Epoch:29, Loss:0.0084
Epoch:30, Loss:0.0083


In [59]:
labels.shape

torch.Size([64])